### Prompt Chaining Usecase Implementation

Usecase:
* to generate and improvize story based on feedback

In [3]:
#importing libraries

import os
from dotenv import load_dotenv

from langchain.chat_models import init_chat_model

load_dotenv()

d:\Agentic_AI\agenticenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [4]:
#base model
model = init_chat_model(model="groq:llama-3.1-8b-instant")
model

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x00000178251DBBF0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000017825360800>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'))

#### With LLM Model

In [5]:
#defining state schema
from typing_extensions import TypedDict

from langgraph.graph import START, END, StateGraph

class StoryState(TypedDict):
    topic: str
    story: str
    improved_story: str
    final_story: str


In [ ]:
#defining story generating node

def generate_story(state:StoryState):
    "this function generates story based on given topic"
    response = model.invoke(f"""generate a simple story about the given topic.
                            Topic: {state['topic']}""")
    return {"story": response.content}

#defining decision node to route based on condition

def judge_Story(state: StoryState):
    "this function judegs the story generated"
    if '?' or '!' in state['story'] :
        return "fail"
    else:
        return "pass"
    
#defining improvizing story node

def improve_story(state:StoryState):
    "this function improves the story generated by LLM"   
    response = model.invoke(f"""improvize the given story to have a happy ending always.
                            story: {state['story']}""")
    return {"improved_story": response.content}

#defining the final polished story node

def polish_story(state:StoryState):
    "this function generates the final story"
    response = model.invoke(f"""summarize the story given in bullet point format.
                            story: {state['improved_story']}""")
    return {"final_story": response.content}

In [ ]:
#defining graph nodes and edges and workflow

graph = StateGraph

#####